In [1]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset --------
class ModeAndFeatureDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, max_items=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))
            if min_len < 15: continue

            own_speeds = np.array([f['OwnSpeed'] / 3.6 for f in seq], dtype=np.float32)
            tgt_speeds = np.array([f['TgtSpeed_ref'] / 3.6 for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                if max_items and len(self.items) >= max_items:
                    return

                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                t = tgt_speeds[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                s2 = np.gradient(s1)  # ⬅️ 自車加速度（2階微分）
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed = np.mean(t - s)
                feature = np.concatenate([modes, d, s, a, s1, s2, d1, d2, rel_acc] + d_smooths)
                self.items.append((feature.astype(np.float32), rel_speed, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, tgt, sid = self.items[idx]
        return torch.tensor(feature), torch.tensor(tgt, dtype=torch.float32), sid

# -------- Collate --------
def collate_fn(batch):
    feats, tgts, sids = zip(*batch)
    return torch.stack(feats), torch.tensor(tgts), list(sids)

# -------- LSTM + Deep Attention --------
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_dim=15, feature_dim=15, hidden_size=128):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)
        self.attn = nn.Sequential(
            nn.Linear(hidden_size, 128),  # 多層化
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, F = x.shape
        x = x.view(B, 15, -1)
        x = x.reshape(-1, x.size(2))
        x = self.pre_fc(x)
        x = x.view(B, 15, -1)

        lstm_out, _ = self.lstm(x)
        attn_weights = self.attn(lstm_out).squeeze(-1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        context = torch.sum(attn_weights.unsqueeze(-1) * lstm_out, dim=1)

        return self.fc_out(context).squeeze(1)

# -------- Training Loop --------
def train_lstm_model(dataset, save_path="model_lstm_attn_extended.pth"):
    scenes = sorted(list(set([item[-1] for item in dataset.items])))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx[:6000])
    val_ds = Subset(dataset, val_idx[:1500])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_dim = train_ds[0][0].shape[0] // 15
    model = ExtendedLSTMWithAttention(input_dim=15, feature_dim=input_dim).to(device)

    def init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    model.apply(init_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 100
    patience_counter = 0

    for epoch in range(200):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行 --------
crop_root = "./train/disparity_crops"
annot_root = "./train/train_annotations"
distance_json_path = "distance_estimates_filtered.json"

dataset = ModeAndFeatureDataset(
    crop_root=crop_root,
    annot_root=annot_root,
    distance_json_path=distance_json_path,
    max_items=7500
)

model = train_lstm_model(dataset, save_path="model_lstm_attn_extended.pth")


[Train 1]: 100%|██████████| 93/93 [00:01<00:00, 80.84it/s] 


Epoch 1 | Train Loss: 0.7921 | Val Loss: 0.4746
✅ Saved model to model_lstm_attn_extended.pth (val_loss=0.4746)


[Train 2]: 100%|██████████| 93/93 [00:00<00:00, 136.40it/s]


Epoch 2 | Train Loss: 0.7019 | Val Loss: 0.2390
✅ Saved model to model_lstm_attn_extended.pth (val_loss=0.2390)


[Train 3]: 100%|██████████| 93/93 [00:00<00:00, 130.66it/s]


...

Epoch 114 | Train Loss: 0.0584 | Val Loss: 0.3416


[Train 115]: 100%|██████████| 93/93 [00:00<00:00, 138.91it/s]


Epoch 115 | Train Loss: 0.0607 | Val Loss: 0.3416


[Train 116]: 100%|██████████| 93/93 [00:00<00:00, 143.91it/s]

Epoch 116 | Train Loss: 0.0591 | Val Loss: 0.3422
🛑 Early stopping at epoch 116
